# Notebook 1 — Population Generation
Generates 10 different synthetic populations with different random seeds.
Run on **Colab CPU**. No GPU needed.
Output: `population_seed{N}.parquet` for N in 0..9, saved to Google Drive.

In [ ]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/epidemic_project'
os.makedirs(f'{BASE}/data', exist_ok=True)
os.makedirs(f'{BASE}/graphs', exist_ok=True)
print('Drive mounted. BASE =', BASE)

In [ ]:
# Cell 2 — Install dependencies
!pip install -q faker pyarrow pandas numpy scikit-learn

In [ ]:
# Cell 3 — Imports
import numpy as np
import pandas as pd
from faker import Faker
from itertools import combinations

# 20 zone centroids around Pune
ZONE_CENTROIDS = {
    1:(18.5204,73.8567), 2:(18.5314,73.8446), 3:(18.5089,73.8741),
    4:(18.4968,73.8631), 5:(18.5421,73.8789), 6:(18.5612,73.8234),
    7:(18.4823,73.9012), 8:(18.5198,73.9156), 9:(18.4712,73.8345),
    10:(18.5534,73.8678),11:(18.4634,73.8912),12:(18.5023,73.8123),
    13:(18.5756,73.8456),14:(18.4891,73.8234),15:(18.5345,73.9234),
    16:(18.5067,73.7987),17:(18.4756,73.9345),18:(18.5589,73.8012),
    19:(18.4623,73.8678),20:(18.5234,73.8901),
}
print('Imports done.')

In [ ]:
# Cell 4 — Population generator function
def generate_population(seed=42, N=12000):
    """
    Generate a synthetic population of N people.
    Each call with a different seed produces a structurally
    different population (different household sizes, workplaces,
    social frequencies, age distributions).
    """
    np.random.seed(seed)
    fake = Faker('en_IN')
    fake.seed_instance(seed)

    # --- Households ---
    # Vary mean household size slightly per seed
    lam = 2.0 + (seed % 5) * 0.2   # lam between 2.0 and 2.8
    n_households = N // 3
    hh_sizes = np.random.poisson(lam=lam, size=n_households)
    hh_sizes = np.clip(hh_sizes, 1, 8)

    # Vary zone density distribution per seed
    zipf_exp = 1.0 + (seed % 4) * 0.15   # 1.0 to 1.45
    zipf_w = np.array([1/z**zipf_exp for z in range(1, 21)])
    zipf_w /= zipf_w.sum()
    hh_zones = np.random.choice(range(1, 21), size=n_households, p=zipf_w)

    node_hh, node_zone = [], []
    for hid, (sz, zn) in enumerate(zip(hh_sizes, hh_zones)):
        for _ in range(sz):
            node_hh.append(hid)
            node_zone.append(zn)
            if len(node_hh) >= N:
                break
        if len(node_hh) >= N:
            break

    node_hh   = np.array(node_hh[:N])
    node_zone = np.array(node_zone[:N])
    N_actual  = len(node_hh)

    # --- Age: mixture of working + elderly, varied per seed ---
    elderly_frac = 0.15 + (seed % 6) * 0.02   # 0.15 to 0.25
    age_work    = np.random.normal(32, 12, N_actual)
    age_elderly = np.random.normal(68,  8, N_actual)
    is_elderly  = np.random.binomial(1, elderly_frac, N_actual).astype(bool)
    age = np.where(is_elderly, age_elderly, age_work)
    age = np.clip(age, 1, 95).astype(int)

    sex = np.random.binomial(1, 0.51, N_actual)

    # --- Socioeconomic: Beta(a,b) varied per seed ---
    ses_a = 1.5 + (seed % 4) * 0.3
    ses   = np.random.beta(ses_a, 5, N_actual)

    # --- Social interaction: Zipf with varying exponent ---
    zipf_social = 1.3 + (seed % 5) * 0.1   # 1.3 to 1.7
    raw = np.random.zipf(zipf_social, N_actual * 3)
    raw = raw[raw <= 60][:N_actual]
    if len(raw) < N_actual:
        raw = np.concatenate([raw, np.ones(N_actual - len(raw), dtype=int)])
    social_freq = raw.astype(float)

    social_norm = (social_freq - social_freq.min()) / (social_freq.max() - social_freq.min() + 1e-8)
    mobility    = 0.6 * social_norm + 0.4 * np.random.uniform(0, 1, N_actual)

    # --- Comorbidity and vaccination ---
    comorbidity = np.random.beta(1.5, 8, N_actual)
    vax_base    = 0.55 + (seed % 5) * 0.04   # 0.55 to 0.71
    vax_prob    = vax_base - 0.2 * (1 - ses)
    vaccinated  = np.random.binomial(1, np.clip(vax_prob, 0.1, 0.95), N_actual)

    # --- Workplaces and schools ---
    n_wp = 300 + seed * 20
    wp_sizes = np.random.lognormal(3.5, 1.2, n_wp).astype(int)
    wp_sizes = np.clip(wp_sizes, 3, 400)
    n_sc = 60 + seed * 5
    sc_sizes = np.random.randint(80, 600, n_sc)

    age_worker  = age >= 18
    age_student = (age >= 6) & (age < 18)
    wp_id = np.full(N_actual, -1)
    sc_id = np.full(N_actual, -1)

    workers = np.where(age_worker)[0]
    np.random.shuffle(workers)
    wid, cnt = 0, 0
    for i in workers:
        wp_id[i] = wid
        cnt += 1
        if wid < n_wp - 1 and cnt >= wp_sizes[wid]:
            wid += 1; cnt = 0

    students = np.where(age_student)[0]
    np.random.shuffle(students)
    sid, cnt = 0, 0
    for i in students:
        sc_id[i] = sid
        cnt += 1
        if sid < n_sc - 1 and cnt >= sc_sizes[sid]:
            sid += 1; cnt = 0

    # --- Coordinates ---
    lats = np.zeros(N_actual)
    lngs = np.zeros(N_actual)
    geo_spread = 0.006 + (seed % 4) * 0.002   # vary geographic spread
    for i in range(N_actual):
        z = node_zone[i]
        clat, clng = ZONE_CENTROIDS[z]
        lats[i] = clat + np.random.normal(0, geo_spread)
        lngs[i] = clng + np.random.normal(0, geo_spread)

    df = pd.DataFrame({
        'node_id':      np.arange(N_actual),
        'household_id': node_hh,
        'zone':         node_zone,
        'workplace_id': wp_id,
        'school_id':    sc_id,
        'age':          age,
        'sex':          sex,
        'ses':          ses,
        'social_freq':  social_freq,
        'mobility':     mobility,
        'comorbidity':  comorbidity,
        'vaccinated':   vaccinated,
        'lat':          lats,
        'lng':          lngs,
        'pop_seed':     seed,   # track which population this came from
    })
    return df

In [ ]:
# Cell 5 — Generate 10 populations
SEEDS = [0, 7, 13, 21, 42, 55, 77, 88, 99, 111]

for seed in SEEDS:
    print(f'Generating population seed={seed}...', end=' ')
    df = generate_population(seed=seed, N=12000)
    out = f'{BASE}/data/population_seed{seed}.parquet'
    df.to_parquet(out, index=False)
    print(f'saved {len(df)} nodes -> {out}')

print('\nAll 10 populations generated.')
print('Files in data/:')
for f in sorted(os.listdir(f'{BASE}/data')):
    print(' ', f)

In [ ]:
# Cell 6 — Quick sanity check on one population
df = pd.read_parquet(f'{BASE}/data/population_seed42.parquet')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print()
print(df[['age','ses','social_freq','comorbidity','vaccinated']].describe().round(3))